In [ ]:
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import zipfile

cv2.__version__

In [ ]:
import tensorflow
tensorflow.__version__

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
diretorio = '/content/gdrive/MyDrive/LAMIA - Bootcamp/card23/pratice/'

In [ ]:
from tensorflow.keras.models import load_model

# mesmo modelo_01.h5 checkpoint com augmentation usado na versao simples
model = load_model(diretorio + 'modelo_02_expressoes.h5')

In [ ]:
arquivo_video = diretorio + 'testes/video_teste.mp4'
cap = cv2.VideoCapture(arquivo_video)
# abre a captura de video

conectado, video = cap.read()
# le o primeiro frame do video
print(video.shape)

In [ ]:
redimensionar = True
largura_maxima = 600

# se redimensiomaneto ativo e a largura do video for maior que a maxima, redimensiona o video
if redimensionar and video.shape[1] > largura_maxima:
    proporcao = video.shape[1] / video.shape[0]
    video_largura = largura_maxima
    video_altura = int(video_largura / proporcao)
else: # se nao matem o tamanho
    video_largura = video.shape[1]
    video_altura = video.shape[0]

In [ ]:
nome_arquivo = diretorio + 'resultado_video_teste_detalhado.avi'

fourcc = cv2.VideoWriter_fourcc(*'XVID')
#estensao para salvar o video
fps = 24
# quantos fps por segundo

saida_video = cv2.VideoWriter(nome_arquivo, fourcc, fps, (video_largura, video_altura))
# cria o objeto para salavar o video

In [ ]:
from tensorflow.keras.preprocessing.image import img_to_array

# modo detalhado desenha as barras de probabilidade de cada emocao no canto da tela
unica_face = True
# se Falso desenha a emocao em cima de todos os rostos detectados, mas sem as barras de probabilidade

haarcascade_faces = diretorio + 'haarcascade_frontalface_default.xml'
# pegga o arquivo de detecção de rosto OpenCV

fonte_pequena, fonte_media = 0.4, 0.7
# define o tamanho da fonte
fonte = cv2.FONT_HERSHEY_SIMPLEX

# ordem dos labels FER
["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]

face_cascade = cv2.CascadeClassifier(haarcascade_faces)


while cv2.waitKey(1) < 0:
    # loop para processar cada frame do video
    conectado, frame = cap.read()

    if not conectado:# se nao leu o frame encerra
        break

    t = time.time()# contagem de tempo para processar o frame

    if redimensionar:
        frame = cv2.resize(frame, (video_largura, video_altura))
        # se for necessario resimensionar o frame chama a funcao

    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)# converte o frame para escala de cinza
    faces = face_cascade.detectMultiScale(cinza, scaleFactor=1.2, minNeighbors=5, minSize=(30, 30))
    # faz a detecção de rostos

    if len(faces) > 0:
        for (x, y, w, h) in faces:

            # de tem mais de um rosto detectado ele pega o maior 
            if unica_face and len(faces) > 1:
                max_area_face = faces[0]
                for face in faces:
                    if face[2] * face[3] > max_area_face[2] * max_area_face[3]:
                        max_area_face = face
                (x, y, w, h) = max_area_face

            frame = cv2.rectangle(frame, (x, y), (x + w, y + h + 10), (255, 50, 50), 2)
            # desenha o retangulo em volta do rosto detectado

            roi = cinza[y:y + h, x:x + w]
            roi = cv2.resize(roi, (48, 48))
            # 48x48 tamanho de entrada do modelo
            roi = roi.astype('float') / 255.0
            roi = img_to_array(roi)
            roi = np.expand_dims(roi, axis=0)

            result = model.predict(roi)[0]
            # predição de label para o frame

            if result is not None:
                # pega o maior label de probabilidade e coloca em texto em cima da face
                if unica_face:
                    for (index, (emotion, prob)) in enumerate(zip(expressoes, result)):
                        text = '{}: {:.2f}%'.format(emotion, prob * 100)
                        barra = int(prob * 150)
                        espaco_esquerda = 7
                        if barra <= espaco_esquerda:
                            barra = espaco_esquerda + 1
                        # a barra cresce da esquerda pra direita conforme a probabilidade da emocao
                        cv2.rectangle(frame, (espaco_esquerda, (index * 18) + 7), (barra, (index * 18) + 18), (200, 250, 20), -1)
                        cv2.putText(frame, text, (15, (index * 18) + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 0, 0), 1, cv2.LINE_AA)

                resultado = np.argmax(result)
                cv2.putText(frame, expressoes[resultado], (x, y - 10), fonte, fonte_media, (255, 255, 255), 1, cv2.LINE_AA)

            if unica_face and len(faces) > 1:
                break
                # ja processou a maior face nao precisa olhar as outras nesse frame

    cv2.putText(frame, ' frame processado em {:.2f} segundos'.format(time.time() - t), (20, video_altura - 20), fonte, fonte_pequena, (250, 250, 250), 0, lineType=cv2.LINE_AA)

    cv2_imshow(frame)
    saida_video.write(frame)

print('Terminou')
saida_video.release()
cv2.destroyAllWindows()